# 🎓 Experimentos com Learned Wavelet (LearnedWaveletDWT1D_QMF)

## Objetivo
Avaliar o impacto de usar wavelets aprendidas (end-to-end) vs wavelets fixas:
- **LearnedWavelet + CNN**
- **LearnedWavelet + LSTM**
- **LearnedWavelet + CNN-LSTM**
- **LearnedWavelet + Transformer**

## Hipótese
Wavelets aprendidas podem adaptar-se às características específicas do sinal,
potencialmente superando wavelets fixas como db2.

## Arquitetura
```
Input (raw signal) -> LearnedWaveletDWT1D_QMF -> [CNN/LSTM/CNN-LSTM/Transformer] -> Output
```

A camada LearnedWaveletDWT1D_QMF aprende os filtros low/high pass durante o treinamento.
Grid search sobre dropout, regularização e tamanho de arquitetura.

In [ ]:
# ── GPU selection (must come BEFORE importing TensorFlow) ──
import os, sys
sys.path.append('.')
sys.path.append('../../models')
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR,
    DL_TRAINING_CONFIG, LEARNED_WAVELET_CONFIG,
    LEARNED_WAVELET_MODELS_CONFIG,
    generate_learned_wavelet_grid, SEED,
    GPU_ID, EPOCHS_OVERRIDE, MAX_GRID_CONFIGS,
)
# (CUDA_VISIBLE_DEVICES já foi configurado pelo experiment_config)

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import warnings
warnings.filterwarnings('ignore')

# TensorFlow (importado APÓS seleção de GPU)
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

# Imports locais - modelos LWT
from LWT import LearnedWaveletDWT1D_QMF, LearnedWaveletPair1D_QMF

from src.models import (
    create_learned_wavelet_cnn_model,
    create_learned_wavelet_lstm_model,
    create_learned_wavelet_cnn_lstm_model,
    create_learned_wavelet_transformer_model,
    get_callbacks, get_distribute_strategy,
)
from src.evaluation import RegressionEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer

# ── Reprodutibilidade: fixar seed global ──
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Multi-GPU strategy
strategy = get_distribute_strategy()

# Configuração
plt.style.use('seaborn-v0_8-whitegrid')
(RESULTS_DIR / "learned_wavelet_experiments").mkdir(parents=True, exist_ok=True)

print(f"\n✅ Imports realizados com sucesso!")
print(f"📦 LearnedWaveletDWT1D_QMF carregado")
print(f"🎲 SEED={SEED} definido para numpy, tf e random")
if GPU_ID:
    print(f"🖥️  GPU selecionada: {GPU_ID}")
if EPOCHS_OVERRIDE:
    print(f"⚡ EPOCHS_OVERRIDE={EPOCHS_OVERRIDE}")
if MAX_GRID_CONFIGS:
    print(f"⚡ MAX_GRID_CONFIGS={MAX_GRID_CONFIGS}")

## 1. Carregar Dados

In [ ]:
# Carregar datasets (raw)
X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

# Adicionar dimensão de canal
print(f"📦 Dados Carregados (Raw multivariado):")
print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

input_shape = X_train.shape[1:]
print(f"\nInput shape: {input_shape}")

## 2. Configuração das Learned Wavelets

In [ ]:
# Configuração da wavelet aprendida
wavelet_config = LEARNED_WAVELET_CONFIG.copy()

print("Configuração LearnedWaveletDWT1D_QMF:")
for k, v in wavelet_config.items():
    print(f"  {k}: {v}")

# Gerenciadores
results_manager = ResultsManager(RESULTS_DIR / "learned_wavelet_experiments")
evaluator = RegressionEvaluator()
visualizer = ExperimentVisualizer()

training_config = DL_TRAINING_CONFIG.copy()

# Armazenar resultados
all_results = {}          # melhor de cada arquitetura
all_histories = {}        # histórico do melhor
all_grid_results = []     # TODOS os resultados do grid
best_models = {}          # melhor modelo Keras de cada arquitetura (para filtros)

## 3. Definição dos Modelos (via src/models.py + config centralizado)

Modelos criados pelo factory em `src/models.py`:
- `create_learned_wavelet_cnn_model`
- `create_learned_wavelet_lstm_model`
- `create_learned_wavelet_cnn_lstm_model` (NOVO)
- `create_learned_wavelet_transformer_model`

Parâmetros base em `LEARNED_WAVELET_MODELS_CONFIG` e grid em `LEARNED_WAVELET_GRID_AXES`.

In [ ]:
# Mapear nome da arquitetura → factory function
MODEL_BUILDERS = {
    'CNN': create_learned_wavelet_cnn_model,
    'LSTM': create_learned_wavelet_lstm_model,
    'CNN_LSTM': create_learned_wavelet_cnn_lstm_model,
    'Transformer': create_learned_wavelet_transformer_model,
}

print("✅ Factory functions mapeadas:")
for name in MODEL_BUILDERS:
    base_cfg = LEARNED_WAVELET_MODELS_CONFIG.get(name, {})
    grid_size = len(generate_learned_wavelet_grid(name))
    print(f"  {name}: {grid_size} combinações no grid")

## 4. Experimento 1: LearnedWavelet + CNN (Grid)

In [ ]:
print("="*70)
print("🎓 Grid Search: LearnedWavelet + CNN")
print("="*70)

arch = 'CNN'
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    # Override wavelet kernel_size from grid
    wc = wavelet_config.copy()
    if 'wavelet_kernel_size' in variation:
        wc['kernel_size'] = variation['wavelet_kernel_size']

    params = {k: v for k, v in {**base_params, **variation}.items() if k != 'wavelet_kernel_size'}
    run_name = f"LearnedWavelet_CNN_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](input_shape, wavelet_config=wc, cnn_params=params)

    model_path = str(MODELS_DIR / f"learned_wavelet_cnn_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        early_patience=training_config['early_stopping_patience'],
        lr_patience=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'LearnedWavelet_CNN', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['LearnedWavelet_CNN'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['LearnedWavelet_CNN'] = history.history
        best_models['LearnedWavelet_CNN'] = model
        model.save(str(MODELS_DIR / "learned_wavelet_cnn_best.keras"))

    results_manager.log_experiment('DL_LearnedWavelet', run_name, metrics,
                                   {'params': params, 'wavelet_config': wc})

print(f"\n🏆 Melhor CNN: {best_key} — RMSE={best_rmse:.6f}")

## 5. Experimento 2: LearnedWavelet + LSTM (Grid)

In [ ]:
print("="*70)
print("🎓 Grid Search: LearnedWavelet + LSTM")
print("="*70)

arch = 'LSTM'
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    # Override wavelet kernel_size from grid
    wc = wavelet_config.copy()
    if 'wavelet_kernel_size' in variation:
        wc['kernel_size'] = variation['wavelet_kernel_size']

    params = {k: v for k, v in {**base_params, **variation}.items() if k != 'wavelet_kernel_size'}
    run_name = f"LearnedWavelet_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](input_shape, wavelet_config=wc, lstm_params=params)

    model_path = str(MODELS_DIR / f"learned_wavelet_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        early_patience=training_config['early_stopping_patience'],
        lr_patience=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'LearnedWavelet_LSTM', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['LearnedWavelet_LSTM'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['LearnedWavelet_LSTM'] = history.history
        best_models['LearnedWavelet_LSTM'] = model
        model.save(str(MODELS_DIR / "learned_wavelet_lstm_best.keras"))

    results_manager.log_experiment('DL_LearnedWavelet', run_name, metrics,
                                   {'params': params, 'wavelet_config': wc})

print(f"\n🏆 Melhor LSTM: {best_key} — RMSE={best_rmse:.6f}")

## 6. Experimento 3: LearnedWavelet + CNN-LSTM (Grid) — NOVO

In [ ]:
print("="*70)
print("🎓 Grid Search: LearnedWavelet + CNN-LSTM")
print("="*70)

arch = 'CNN_LSTM'
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    # Override wavelet kernel_size from grid
    wc = wavelet_config.copy()
    if 'wavelet_kernel_size' in variation:
        wc['kernel_size'] = variation['wavelet_kernel_size']

    params = {k: v for k, v in {**base_params, **variation}.items() if k != 'wavelet_kernel_size'}
    run_name = f"LearnedWavelet_CNN_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](input_shape, wavelet_config=wc, cnn_lstm_params=params)

    model_path = str(MODELS_DIR / f"learned_wavelet_cnn_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        early_patience=training_config['early_stopping_patience'],
        lr_patience=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'LearnedWavelet_CNN_LSTM', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['LearnedWavelet_CNN_LSTM'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['LearnedWavelet_CNN_LSTM'] = history.history
        best_models['LearnedWavelet_CNN_LSTM'] = model
        model.save(str(MODELS_DIR / "learned_wavelet_cnn_lstm_best.keras"))

    results_manager.log_experiment('DL_LearnedWavelet', run_name, metrics,
                                   {'params': params, 'wavelet_config': wc})

print(f"\n🏆 Melhor CNN-LSTM: {best_key} — RMSE={best_rmse:.6f}")

## 7. Experimento 4: LearnedWavelet + Transformer (Grid)

In [ ]:
print("="*70)
print("🎓 Grid Search: LearnedWavelet + Transformer")
print("="*70)

arch = 'Transformer'
grid = generate_learned_wavelet_grid(arch)
base_params = LEARNED_WAVELET_MODELS_CONFIG[arch].copy()
best_rmse, best_key = float('inf'), None

for gi, variation in enumerate(grid):
    # Override wavelet kernel_size from grid
    wc = wavelet_config.copy()
    if 'wavelet_kernel_size' in variation:
        wc['kernel_size'] = variation['wavelet_kernel_size']

    params = {k: v for k, v in {**base_params, **variation}.items() if k != 'wavelet_kernel_size'}
    run_name = f"LearnedWavelet_Transformer_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = MODEL_BUILDERS[arch](input_shape, wavelet_config=wc, transformer_params=params)

    model_path = str(MODELS_DIR / f"learned_wavelet_transformer_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        early_patience=training_config['early_stopping_patience'],
        lr_patience=training_config['reduce_lr_patience'],
        min_lr=training_config['min_lr'],
        use_reduce_lr=not params.get('use_warmup', True),
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config['epochs'],
        batch_size=training_config['batch_size'],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test, verbose=0).flatten()
    metrics = evaluator.evaluate(y_test, y_pred)

    print(f"    RMSE={metrics['rmse']:.6f}  MAE={metrics['mae']:.6f}  "
          f"R²={metrics['r2']:.6f}  Time={elapsed:.1f}s")

    row = {'Model': 'LearnedWavelet_Transformer', 'grid_idx': gi, **variation,
           'RMSE': metrics['rmse'], 'MAE': metrics['mae'],
           'R²': metrics['r2'], 'Params': model.count_params(),
           'Time (s)': elapsed, 'Epochs': len(history.history['loss'])}
    all_grid_results.append(row)

    if metrics['rmse'] < best_rmse:
        best_rmse = metrics['rmse']
        best_key = run_name
        all_results['LearnedWavelet_Transformer'] = {
            'metrics': metrics, 'time': elapsed,
            'epochs': len(history.history['loss']),
            'y_pred': y_pred, 'params': model.count_params(),
            'best_variation': variation,
        }
        all_histories['LearnedWavelet_Transformer'] = history.history
        best_models['LearnedWavelet_Transformer'] = model
        model.save(str(MODELS_DIR / "learned_wavelet_transformer_best.keras"))

    results_manager.log_experiment('DL_LearnedWavelet', run_name, metrics,
                                   {'params': params, 'wavelet_config': wc})

print(f"\n🏆 Melhor Transformer: {best_key} — RMSE={best_rmse:.6f}")

## 8. Comparação dos Resultados

In [ ]:
# Criar DataFrame comparativo (melhor de cada arquitetura)
comparison_data = []
for model_name, result in all_results.items():
    row = {
        'Model': model_name,
        'RMSE': result['metrics']['rmse'],
        'MAE': result['metrics']['mae'],
        'R²': result['metrics']['r2'],
        'Params': result['params'],
        'Time (s)': result['time'],
        'Epochs': result['epochs']
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('RMSE')

print("\n" + "="*70)
print("📊 COMPARAÇÃO - Modelos com LearnedWaveletDWT1D_QMF (melhor por arquitetura)")
print("="*70)
print(comparison_df.to_string(index=False))

# Salvar comparação (melhor por modelo)
out_dir = RESULTS_DIR / "learned_wavelet_experiments"
comparison_df.to_csv(out_dir / "comparison_learned_wavelet.csv", index=False)

# Salvar TODOS os resultados do grid
grid_df = pd.DataFrame(all_grid_results)
grid_df.to_csv(out_dir / "all_grid_results_learned_wavelet.csv", index=False)
print(f"\n✅ CSV comparação salvo: {out_dir / 'comparison_learned_wavelet.csv'}")
print(f"✅ CSV grid completo salvo: {out_dir / 'all_grid_results_learned_wavelet.csv'}")
print(f"   Total de combinações treinadas: {len(grid_df)}")

In [ ]:
# Visualização comparativa
n_models = len(comparison_df)
fig, axes = plt.subplots(1, 3, figsize=(16, max(4, n_models * 1.2)))

metrics_to_plot = ['RMSE', 'MAE', 'R²']
colors = plt.cm.Purples(np.linspace(0.4, 0.9, n_models))

for idx, metric in enumerate(metrics_to_plot):
    data = comparison_df.set_index('Model')[metric].sort_values(
        ascending=(metric != 'R²')
    )
    bars = axes[idx].barh(data.index, data.values, color=colors[:len(data)])
    axes[idx].set_xlabel(metric)
    axes[idx].set_title(f'Comparação: {metric}')
    axes[idx].grid(True, alpha=0.3, axis='x')
    
    for bar, val in zip(bars, data.values):
        axes[idx].text(val, bar.get_y() + bar.get_height()/2,
                      f'{val:.4f}', va='center', ha='left', fontsize=9)

plt.suptitle('Learned Wavelet (LearnedWaveletDWT1D_QMF) — 4 Arquiteturas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learned_wavelet_experiments" / "comparison_learned_wavelet.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Evolução do Treinamento

Visualização detalhada da evolução do processo de treinamento para cada arquitetura com Learned Wavelets:
- **Loss** (Train vs Validation) ao longo das épocas
- **Convergência** — velocidade e estabilidade
- **Early Stopping** — ponto de parada otimizado

In [ ]:
# ── Evolução do Treinamento: Loss (Train vs Val) por modelo ──
n_models = len(all_histories)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

colors_lines = {'loss': '#2196F3', 'val_loss': '#F44336'}

for idx, (model_name, history) in enumerate(all_histories.items()):
    ax = axes[idx]
    epochs_range = range(1, len(history['loss']) + 1)
    
    # Train & Val loss
    ax.plot(epochs_range, history['loss'], color=colors_lines['loss'],
            linewidth=2, label='Train Loss', alpha=0.9)
    ax.plot(epochs_range, history['val_loss'], color=colors_lines['val_loss'],
            linewidth=2, label='Val Loss', alpha=0.9)
    
    # Marcar melhor época (menor val_loss)
    best_epoch = np.argmin(history['val_loss']) + 1
    best_val = min(history['val_loss'])
    ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch={best_epoch}')
    ax.scatter([best_epoch], [best_val], color='green', s=100, zorder=5, edgecolors='black')
    
    # Anotação
    ax.annotate(f'Val Loss={best_val:.6f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + len(epochs_range)*0.05, best_val * 1.15),
                fontsize=9, color='green',
                arrowprops=dict(arrowstyle='->', color='green', alpha=0.7))
    
    ax.set_xlabel('Época', fontsize=11)
    ax.set_ylabel('Loss (MSE)', fontsize=11)
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

# Esconder subplots não usados
for idx in range(n_models, 4):
    axes[idx].set_visible(False)

plt.suptitle('Evolução do Treinamento — Learned Wavelet (Loss por Época)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learned_wavelet_experiments" / "training_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Comparativo: todas as curvas de val_loss sobrepostas ──
fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab10
for idx, (model_name, history) in enumerate(all_histories.items()):
    epochs_range = range(1, len(history['val_loss']) + 1)
    ax.plot(epochs_range, history['val_loss'], linewidth=2, label=model_name,
            color=cmap(idx), alpha=0.85)
    best_ep = np.argmin(history['val_loss']) + 1
    best_vl = min(history['val_loss'])
    ax.scatter([best_ep], [best_vl], color=cmap(idx), s=80, zorder=5, edgecolors='black')

ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('Validation Loss (MSE)', fontsize=12)
ax.set_title('Comparativo — Convergência Val Loss (Learned Wavelet)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "learned_wavelet_experiments" / "val_loss_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Resumo numérico da evolução ──
print("\n📊 Resumo da Evolução do Treinamento:")
print(f"{'Modelo':<30} {'Épocas':>7} {'Train Loss Final':>16} {'Val Loss Final':>15} {'Best Val Loss':>14} {'Best Epoch':>11}")
print("-" * 100)
for model_name, history in all_histories.items():
    n_ep = len(history['loss'])
    best_ep = np.argmin(history['val_loss']) + 1
    print(f"{model_name:<30} {n_ep:>7} {history['loss'][-1]:>16.6f} {history['val_loss'][-1]:>15.6f} {min(history['val_loss']):>14.6f} {best_ep:>11}")

## 10. Visualização dos Filtros Aprendidos

In [ ]:
# Extrair e visualizar filtros aprendidos do melhor modelo CNN
def extract_learned_filters(model):
    """Extrai os filtros aprendidos da camada wavelet."""
    for layer in model.layers:
        if 'learned_wavelet' in layer.name.lower():
            pairs = layer.pairs
            filters_info = []
            for i, pair in enumerate(pairs):
                t = pair._make_t()
                scale = tf.nn.softplus(pair.raw_scale) + 1e-3
                t_adj = (t - pair.translation) / scale
                z = pair.base_net(t_adj)
                h = pair.low_head(z)
                h = pair._normalize_h(h)
                g = pair._qmf_from_h(h)
                filters_info.append({
                    'level': i + 1,
                    'low_pass': h.numpy().flatten(),
                    'high_pass': g.numpy().flatten(),
                    'scale': scale.numpy(),
                    'translation': pair.translation.numpy()
                })
            return filters_info
    return None

# Usar o melhor modelo CNN para visualização
ref_model = best_models.get('LearnedWavelet_CNN')
if ref_model is not None:
    filters = extract_learned_filters(ref_model)
    if filters:
        n_levels = len(filters)
        fig, axes = plt.subplots(n_levels, 2, figsize=(14, 4*n_levels))
        if n_levels == 1:
            axes = axes[np.newaxis, :]
        for i, filt in enumerate(filters):
            axes[i, 0].plot(filt['low_pass'], 'b-', linewidth=2)
            axes[i, 0].set_title(f'Nível {filt["level"]} - Filtro Low-Pass (h)')
            axes[i, 0].grid(True, alpha=0.3)
            axes[i, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
            axes[i, 1].plot(filt['high_pass'], 'r-', linewidth=2)
            axes[i, 1].set_title(f'Nível {filt["level"]} - Filtro High-Pass (g) [QMF]')
            axes[i, 1].grid(True, alpha=0.3)
            axes[i, 1].axhline(y=0, color='b', linestyle='--', alpha=0.5)
        plt.suptitle('Filtros Wavelet Aprendidos (LearnedWaveletDWT1D_QMF)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "learned_wavelet_experiments" / "learned_filters.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("\n📊 Parâmetros dos Filtros Aprendidos:")
        for filt in filters:
            print(f"  Nível {filt['level']}: scale={filt['scale']:.4f}, translation={filt['translation']:.4f}")
else:
    print("⚠️ Nenhum modelo CNN best_model disponível para visualização de filtros")

## 11. Resumo

In [ ]:
print("\n" + "="*70)
print("📋 RESUMO - Experimentos com Learned Wavelets")
print("="*70)
print(f"\n✅ Arquiteturas avaliadas: {len(comparison_df)} (CNN, LSTM, CNN-LSTM, Transformer)")
print(f"✅ Total de variações de grid testadas: {len(all_grid_results)}")
best_row = comparison_df.sort_values('RMSE').iloc[0]
print(f"✅ Melhor modelo: {best_row['Model']}")
print(f"✅ Melhor RMSE: {best_row['RMSE']:.6f}")
print(f"✅ Melhor R²: {best_row['R²']:.6f}")
print(f"\n📁 Resultados salvos em: {RESULTS_DIR / 'learned_wavelet_experiments'}")
print("\n🎉 Notebook concluído com sucesso!")